In [2]:
import duckdb
import pandas as pd

# Connect in write mode (we'll create new tables)
con = duckdb.connect('../data/airbnb.db')
print("Connected to DuckDB")

# Verify raw tables still exist
tables = con.execute("SHOW TABLES").fetchall()
print(f"Existing tables: {[t[0] for t in tables]}")

Connected to DuckDB
Existing tables: ['listings', 'listings_clean', 'reviews']


Dropped "listings_clean" temporary view

In [5]:
con.execute("DROP VIEW IF EXISTS listings_clean")
con.execute("DROP TABLE IF EXISTS listings_final")
print("Cleaned up")

tables = con.execute("SHOW TABLES").fetchall()
print(f"Remaining: {[t[0] for t in tables]}")

Cleaned up
Remaining: ['listings', 'reviews']


Standardized "price" column

In [15]:
con.execute("""
    CREATE OR REPLACE TABLE listings_clean AS
    SELECT 
        *,
        TRY_CAST(REPLACE(REPLACE(REPLACE(price, '$', ''), ',', ''), ' ', '') AS DECIMAL) as price_clean,
        TRY_CAST(REPLACE(host_response_rate, '%', '') AS INTEGER) as host_response_rate_clean,
        TRY_CAST(REPLACE(host_acceptance_rate, '%', '') AS INTEGER) as host_acceptance_rate_clean
    FROM listings
""")
print("Created listings_clean")

# Verify
result = con.execute("""
    SELECT price as original, price_clean 
    FROM listings_clean 
    LIMIT 5
""").df()
print(result)


Created listings_clean
  original  price_clean
0   $59.00         59.0
1  $120.00        120.0
2  $493.00        493.0
3  $190.00        190.0
4  $140.00        140.0


Verification of price standardizing step

In [19]:
print("=" * 60)
print("VERIFICATION: Price & Rate Conversions")
print("=" * 60)

# 1. Comma prices
comma_check = con.execute("""
    SELECT 
        COUNT(*) as total_with_commas,
        SUM(CASE WHEN price_clean IS NOT NULL THEN 1 ELSE 0 END) as successfully_converted
    FROM listings_clean
    WHERE price LIKE '%,%'
""").df()
print("\n Prices WITH commas:")
print(comma_check.to_string(index=False))

# 2. N/A in response rate
na_response = con.execute("""
    SELECT 
        COUNT(*) as total_na,
        SUM(CASE WHEN host_response_rate_clean IS NULL THEN 1 ELSE 0 END) as became_null
    FROM listings_clean
    WHERE host_response_rate = 'N/A'
""").df()
print("\n host_response_rate = 'N/A':")
print(na_response.to_string(index=False))

# 3. N/A in acceptance rate
na_acceptance = con.execute("""
    SELECT 
        COUNT(*) as total_na,
        SUM(CASE WHEN host_acceptance_rate_clean IS NULL THEN 1 ELSE 0 END) as became_null
    FROM listings_clean
    WHERE host_acceptance_rate = 'N/A'
""").df()
print("\n host_acceptance_rate = 'N/A':")
print(na_acceptance.to_string(index=False))

# 4. Sample: comma prices
print("\n" + "=" * 60)
print("SAMPLE: Comma prices converted")
print("=" * 60)
comma_samples = con.execute("""
    SELECT price as original, price_clean 
    FROM listings_clean 
    WHERE price LIKE '%,%'
    LIMIT 5
""").df()
print(comma_samples.to_string(index=False))

# 5. Sample: response rate conversions
print("\n" + "=" * 60)
print("SAMPLE: Response rate conversions")
print("=" * 60)
rate_samples = con.execute("""
    SELECT 
        host_response_rate as original, 
        host_response_rate_clean as cleaned
    FROM listings_clean 
    WHERE host_response_rate IS NOT NULL
    LIMIT 10
""").df()
print(rate_samples.to_string(index=False))

VERIFICATION: Price & Rate Conversions

 Prices WITH commas:
 total_with_commas  successfully_converted
               715                   715.0

 host_response_rate = 'N/A':
 total_na  became_null
    31876      31876.0

 host_acceptance_rate = 'N/A':
 total_na  became_null
    27221      27221.0

SAMPLE: Comma prices converted
 original  price_clean
$1,300.00       1300.0
$1,500.00       1500.0
$2,000.00       2000.0
$1,134.00       1134.0
$1,000.00       1000.0

SAMPLE: Response rate conversions
original  cleaned
    100%    100.0
     N/A      NaN
    100%    100.0
    100%    100.0
     N/A      NaN
    100%    100.0
    100%    100.0
    100%    100.0
     N/A      NaN
    100%    100.0


Identifing and capping extreme price-outliers

In [24]:
con.execute("""
    CREATE OR REPLACE TABLE listings_clean AS
    SELECT 
        *,
        CASE 
            WHEN price_clean > 5000 THEN NULL  -- Cap at $5000/night
            WHEN price_clean < 10 THEN NULL    -- Flag suspiciously low
            ELSE price_clean
        END as price_final,
        -- Flag for review
        CASE 
            WHEN price_clean > 5000 THEN 'EXTREME_HIGH'
            WHEN price_clean < 10 THEN 'EXTREME_LOW'
            ELSE 'NORMAL'
        END as price_flag
    FROM listings_clean
""")
print("Added price_final and price_flag columns")

# Check how many were flagged
flag_stats = con.execute("""
    SELECT price_flag, COUNT(*) as count
    FROM listings_clean
    GROUP BY price_flag
""").df()
print("\n Price flag distribution:")
print(flag_stats.to_string(index=False))

Added price_final and price_flag columns

 Price flag distribution:
  price_flag  count
      NORMAL  96128
EXTREME_HIGH     47
 EXTREME_LOW      7


Date field validation

In [23]:
date_columns = con.execute("""
    SELECT column_name, column_type
    FROM (SUMMARIZE listings_clean)
    WHERE column_type = 'DATE'
""").df()
print("DATE COLUMNS:")
print(date_columns.to_string(index=False))

date_ranges = con.execute("""
    SELECT 
        MIN(last_scraped) as min_scrape, MAX(last_scraped) as max_scrape,
        MIN(first_review) as min_first_review, MAX(first_review) as max_first_review
    FROM listings_clean
""").df()
print("\n Date ranges:")
print(date_ranges.to_string(index=False))

DATE COLUMNS:
          column_name column_type
         last_scraped        DATE
           host_since        DATE
calendar_last_scraped        DATE
         first_review        DATE
          last_review        DATE

 Date ranges:
min_scrape max_scrape min_first_review max_first_review
2024-09-06 2024-09-11       2009-12-21       2024-09-06


Normalized text fields

In [25]:
room_types = con.execute("""
    SELECT room_type, COUNT(*) as count
    FROM listings_clean GROUP BY room_type ORDER BY count DESC
""").df()
print("ROOM TYPES:")
print(room_types.to_string(index=False))

ROOM TYPES:
      room_type  count
Entire home/apt  61321
   Private room  34236
    Shared room    437
     Hotel room    188


Missing value handling

In [26]:
con.execute("""
    CREATE OR REPLACE TABLE listings_clean AS
    SELECT 
        *,
        COALESCE(neighbourhood, neighbourhood_cleansed, 'Unknown') as neighbourhood_final
    FROM listings_clean
""")

nb_check = con.execute("""
    SELECT 
        SUM(CASE WHEN neighbourhood IS NULL THEN 1 ELSE 0 END) as original_nulls,
        SUM(CASE WHEN neighbourhood_final = 'Unknown' THEN 1 ELSE 0 END) as still_unknown
    FROM listings_clean
""").df()
print("NEIGHBOURHOOD IMPUTATION:")
print(nb_check.to_string(index=False))

con.execute("""
    ALTER TABLE listings_clean ADD COLUMN superhost_status VARCHAR;
    UPDATE listings_clean 
    SET superhost_status = CASE 
        WHEN host_is_superhost IS NULL THEN 'unknown'
        WHEN host_is_superhost = TRUE THEN 't'
        ELSE 'f'
    END;
""")
print("Added superhost_status column")

NEIGHBOURHOOD IMPUTATION:
 original_nulls  still_unknown
        50520.0            0.0
Added superhost_status column


Flag invalid records

In [27]:
unavailable = con.execute("""
    SELECT COUNT(*) as cnt
    FROM listings_clean
    WHERE name ILIKE '%no longer available%' OR name ILIKE '%unavailable%'
""").df()
print(f"Unavailable: {unavailable['cnt'].iloc[0]:,}")

con.execute("""
    ALTER TABLE listings_clean ADD COLUMN is_active BOOLEAN DEFAULT TRUE;
    UPDATE listings_clean 
    SET is_active = FALSE
    WHERE name ILIKE '%no longer available%' OR name ILIKE '%unavailable%';
""")
print("Flagged unavailable listings")


Unavailable: 12
Flagged unavailable listings


Standardized geography

In [28]:
con.execute("""
    CREATE OR REPLACE TABLE listings_clean AS
    SELECT 
        *,
        ROUND(latitude, 5) as latitude_clean,
        ROUND(longitude, 5) as longitude_clean
    FROM listings_clean
""")
print("Rounded coordinates")

geo = con.execute("""
    SELECT 
        MIN(latitude_clean) as min_lat, MAX(latitude_clean) as max_lat,
        MIN(longitude_clean) as min_lon, MAX(longitude_clean) as max_lon
    FROM listings_clean WHERE is_active = TRUE
""").df()
print(geo.to_string(index=False))


Rounded coordinates
 min_lat  max_lat  min_lon  max_lon
51.29594 51.68164  -0.4978  0.29573


Final cleaned table

In [29]:
con.execute("""
    CREATE OR REPLACE TABLE listings_final AS
    SELECT 
        id, name, description, host_id, host_name, host_since, host_location,
        neighbourhood_cleansed as neighbourhood,
        neighbourhood_final,
        latitude_clean as latitude,
        longitude_clean as longitude,
        room_type, property_type,
        price_final as price,
        price_flag,
        minimum_nights, maximum_nights, availability_365,
        number_of_reviews, number_of_reviews_ltm, reviews_per_month,
        first_review, last_review,
        review_scores_rating, review_scores_accuracy, review_scores_cleanliness,
        review_scores_checkin, review_scores_communication,
        review_scores_location, review_scores_value,
        host_is_superhost, superhost_status,
        host_response_rate_clean as host_response_rate,
        host_acceptance_rate_clean as host_acceptance_rate,
        calculated_host_listings_count,
        instant_bookable, is_active
    FROM listings_clean
    WHERE price_final IS NOT NULL
""")

final_count = con.execute("SELECT COUNT(*) FROM listings_final").fetchone()[0]
print(f"Final listings: {final_count:,}")
print(f"Removed: {96182 - final_count:,} with invalid prices")


Final listings: 63,151
Removed: 33,031 with invalid prices
